# L7a: Dimensionality Reduction
High-dimensional data is everywhere in modern machine learning, from genomics datasets with thousands of features to images with millions of pixels. Dimensionality reduction techniques are tools to compress this data while preserving its most important structure and information.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Compute and interpret the empirical covariance matrix:__ The empirical covariance matrix summarizes how pairs of features vary together across a dataset. We'll build it from the centered data matrix and interpret its eigenvectors as the directions of greatest variation and its eigenvalues as the variance along those directions.
> * __Factor a matrix with the singular value decomposition:__ The singular value decomposition (SVD) factors any real matrix into two orthogonal matrices and a rectangular diagonal matrix of singular values. We'll compare its full, compact, and thin forms and truncate it to get the closest matrix of a given lower rank, which the Eckart–Young theorem guarantees in the Frobenius and spectral norms.
> * __Connect principal component analysis to the SVD:__ Principal component analysis (PCA) reduces dimension by projecting the centered data onto the eigenvectors of the covariance matrix with the largest eigenvalues. We'll show that the SVD of the centered data matrix gives the same directions and variances without forming the covariance matrix.

We'll demonstrate these ideas with an example, where we decompose a grayscale photograph with the SVD and rebuild it from a few rank-one frames. Let's get started!
___

## Examples
We will use the following example to see what the truncated SVD does to real data:

> [▶ Fun with Singular Value Decomposition](CHEME-5800-L7a-Example-FunWithSVD-Fall-2026.ipynb). What does the SVD reveal about the structure of an image? We factor a grayscale photograph into rank-one frames, each weighted by a singular value, and rebuild the image from the first few frames to see how quickly the relative error falls. We also check that the sum of the first few frames has rank equal to the number of frames added.

___

## Dimensionality Reduction Problem
Suppose we have a dataset $\mathcal{D} = \left\{\mathbf{x}_{1},\mathbf{x}_{2},\dots,\mathbf{x}_{n}\right\}$ where $\mathbf{x}_{i}\in\mathbb{R}^{m}$ is an $m$-dimensional feature vector that we want to compress into $k$ dimensions: $\mathbf{x}_i \in \mathbb{R}^m \;\rightarrow\; \mathbf{y}_i \in \mathbb{R}^k$ where $k\ll{m}$.

> **Why does this matter?** Dimensionality reduction shows up everywhere in real applications:
> * **Genomics:** Gene expression data might have 20,000+ features (genes) but only hundreds of samples. Reducing to key patterns reveals biological processes.
> * **Image compression:** A one-megapixel grayscale image has about a million dimensions, one per pixel, but most of the information concentrates in far fewer components (like we'll see with SVD!).
> * **Recommendation systems:** User-item matrices are huge and sparse. Low-dimensional representations capture latent preferences efficiently. The winning solution to Netflix's famous \$1M Prize used matrix factorization techniques, close relatives of the truncated SVD, to improve movie recommendations ([Koren, Bell, and Volinsky, 2009](https://doi.org/10.1109/MC.2009.263)).
> * **Visualization:** You can't plot 1000-dimensional data, but you can visualize the top 2–3 components to spot patterns and clusters.

**Composite features:**  Each component of the lower-dimensional vector $\mathbf{y}_i$ is called a *composite feature*, since it's a linear combination of the original features after subtracting their means.  Reducing dimensionality can help us visualize high-dimensional data in 2–3D, reduce the computational complexity of a machine learning algorithm, or give us a more compact representation of the data that retains the most important information.

Imagine we have a _magical transformation matrix_ $\mathbf{P}\in\mathbb{R}^{k\times{m}}$ that maps a feature vector $\mathbf{x}$ to its composite features:

$$
\mathbf{y} = \mathbf{P}\;(\mathbf{x} - \bar{\mathbf{x}}),
$$

where $\mathbf{y}\in\mathbb{R}^{k}$ is the new composite feature vector and $\bar{\mathbf{x}}$ is the mean of the feature vectors in the dataset. If we write $\mathbf{P} = [\,\boldsymbol{\phi}_1^\top;\dots;\boldsymbol{\phi}_k^\top]$, then each row $\boldsymbol{\phi}_j^\top$ extracts one component:

$$
y_{j} = \boldsymbol{\phi}_{j}^{\top}\;(\mathbf{x} - \bar{\mathbf{x}}),\quad{j=1,2,\dots,k}.
$$

Wow, that sounds great!  What are these magical transformation vectors $\boldsymbol{\phi}_{j}$?
* __TL;DR.__ The $\boldsymbol{\phi}_{j}$ vectors are the top-$k$ eigenvectors of the data's covariance matrix, and this reduction procedure has a special name, it is known as __Principal Component Analysis (PCA)__.
* **Alternative story.** Equivalently, you can compute the transformation matrix $\mathbf{P}$ via another technique called the Singular Value Decomposition (SVD) of the centered data matrix. In this case, the transformation vectors are the top $k$ right singular vectors (the first $k$ columns of the matrix $\mathbf{V}$).

__Hmmm__: There are a few items that we need to introduce before we can get to the SVD story, so let's start with a quick review of the covariance matrix.

___

## Empirical Covariance Matrix
The covariance matrix is a key concept in statistics and machine learning that describes the relationships between different features in a dataset.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-Covariance-Schematic.svg" width="880" alt="Three columns show samples with negative, zero, and positive covariance. In the top row each blue cloud carries a solid one standard deviation ellipse and a dashed two standard deviation ellipse whose tilt follows the sign of the covariance. The bottom row overlays each blue cloud with a red cloud whose covariance matrix is four times larger and draws the dashed two standard deviation ellipse of each cloud; the red ellipse is the blue cloud's ellipse scaled by two, so the standard deviations double while the correlation is preserved."/>
    </center>
</div>

__Sign and scale of covariance.__ Each panel shows samples of two features with negative, zero, or positive covariance. In the upper row, the solid and dashed ellipses mark one and two standard deviations, and their tilt shows the sign of the covariance. In the lower row, the red cloud's covariance matrix is four times that of the blue cloud: the standard deviations double, the correlation is unchanged, and the dashed ellipse scales by two.

Let's start with the covariance between a pair of features, then assemble the full matrix from the data.

Suppose we have a dataset $\mathcal{D} = \left\{\mathbf{x}_{1},\mathbf{x}_{2},\dots,\mathbf{x}_{n}\right\}$ of $n$ samples, where $\mathbf{x}_{k}\in\mathbb{R}^{m}$ is the vector of $m$ features for sample $k$. The empirical covariance matrix $\hat{\mathbf{\Sigma}}\in\mathbb{R}^{m\times m}$ is a square symmetric matrix that summarizes the pairwise covariances between the $m$ features in the dataset $\mathcal{D}$.

Let $x_k^{(i)}$ be the $i$-th feature of sample $k$. Collect the values for feature $i$ into the vector $\mathbf{x}^{(i)}=[x_1^{(i)},\dots,x_n^{(i)}]^\top$. Then, the covariance between features $i$ and $j$ is given by: 
$$
\begin{align*}
    \hat{\Sigma}_{ij} &= \frac{1}{n-1}\sum_{k=1}^{n}\bigl(x^{(i)}_k-\bar{x}_i\bigr)\,\bigl(x^{(j)}_k-\bar{x}_j\bigr) \quad\Longrightarrow\quad\boxed{
\hat\Sigma_{ij}=\sigma_i\,\sigma_j\,\rho_{ij}}\\
\end{align*}
$$
where the mean $\bar{x}_i = \frac{1}{n}\sum_{k=1}^{n}x_k^{(i)}$ is the average of feature $i$ across all $n$ samples, the term $\sigma_{i} = \sqrt{\hat{\Sigma}_{ii}}$ denotes the standard deviation for feature $i$, and $\rho_{ij}\in\left[-1,1\right]$ denotes the correlation between features $i$ and $j$ in the dataset $\mathcal{D}$. But where does this come from? Let's break it down a bit more, starting with the definition of the covariance between features $i$ and $j$:
$$
\begin{align*}
\hat{\Sigma}_{ij} &= \frac{1}{n-1}\sum_{k=1}^{n}\overbrace{\bigl(x^{(i)}_k-\bar{x}_i\bigr)}^{\text{deviation from mean}}\,\bigl(x^{(j)}_k-\bar{x}_j\bigr)\\
\sigma_i^2 & = \hat\Sigma_{ii}=\frac{1}{n-1}\sum_{k=1}^{n}(x_k^{(i)}-\bar x_i)^2,\quad\sigma_j^2=\hat\Sigma_{jj} = \frac{1}{n-1}\sum_{k=1}^{n}(x_k^{(j)}-\bar x_j)^2\\
\rho_{ij} &= \frac{\displaystyle
  \overbrace{\frac{1}{n-1}\sum_{k=1}^n\bigl(x^{(i)}_k-\bar{x}_i\bigr)\,\bigl(x^{(j)}_k-\bar{x}_j\bigr)}^{\hat\Sigma_{ij}}}{
  \underbrace{\sqrt{\frac{1}{n-1}\sum_{k=1}^n\bigl(x^{(i)}_k-\bar{x}_i\bigr)^2}}_{\sigma_i}
  \;\underbrace{\sqrt{\frac{1}{n-1}\sum_{k=1}^n\bigl(x^{(j)}_k-\bar{x}_j\bigr)^2}}_{\sigma_j}
} = \frac{\hat\Sigma_{ij}}{\sigma_i\,\sigma_j}  \quad\Longrightarrow\quad \boxed{\hat\Sigma_{ij} = \sigma_i\,\sigma_j\,\rho_{ij}\quad\blacksquare}
\end{align*}
$$ 

In practice, we compute the entire covariance matrix $\hat{\mathbf{\Sigma}}$ at once from the data matrix $\mathbf{X} \in\mathbb{R}^{n \times m}$, whose rows are samples and whose columns are features; row $k$ contains the values of all $m$ features for sample $k$:
$$
\mathbf{X} = \begin{bmatrix}
x_1^{(1)} & x_1^{(2)} & \cdots & x_1^{(m)} \\
x_2^{(1)} & x_2^{(2)} & \cdots & x_2^{(m)} \\
\vdots & \vdots & \ddots & \vdots \\
x_n^{(1)} & x_n^{(2)} & \cdots & x_n^{(m)}
\end{bmatrix}
$$
To center the data, we need to subtract the mean for each feature. Let $\mathbf{m} = [\bar{x}_1, \bar{x}_2, \ldots, \bar{x}_m]^{\top}$ be the vector containing the mean for each feature. The centered data matrix is given by:
$$
\tilde{\mathbf{X}} = \mathbf{X} - \mathbf{1}\mathbf{m}^{\top}
$$
where $\mathbf{1} \in \mathbb{R}^{n}$ is a vector of ones, and $\mathbf{1}\mathbf{m}^{\top}$ creates an $n \times m$ matrix where each row is identical and contains the means.

> __Outer product:__ The $\mathbf{1}\mathbf{m}^{\top}$ is an example of an outer product. The [outer product](https://en.wikipedia.org/wiki/Outer_product) of two vectors $\mathbf{a} \in \mathbb{R}^{n}$ and $\mathbf{b} \in \mathbb{R}^{m}$ is the $n \times m$ matrix $\mathbf{a}\mathbf{b}^{\top}$. Each element of the outer product is computed as $(\mathbf{a}\mathbf{b}^{\top})_{ij} = a_i b_j$. 

The empirical covariance matrix is then given by:
$$
\hat{\mathbf{\Sigma}} = \frac{1}{n-1}\tilde{\mathbf{X}}^{\top}\tilde{\mathbf{X}}
$$

> __Covariance Matrix Properties:__
>
> The covariance matrix $\hat{\mathbf{\Sigma}}$ has the following (important) properties:
> * __Elements__: The diagonal elements of the covariance matrix $\hat{\Sigma}_{ii}\in\hat{\mathbf{\Sigma}}$ are the variances of feature $i$ (always non-negative), while the off-diagonal elements $\hat{\Sigma}_{ij}\in\hat{\mathbf{\Sigma}}$ for $i\neq{j}$ measure the covariance between features $i$ and $j$ in the dataset $\mathcal{D}$. The sign of $\hat{\Sigma}_{ij}$ gives the direction of the linear relationship between features $i$ and $j$. Its magnitude also depends on the spread of each feature, so the correlation $\rho_{ij}$ measures the strength of the relationship.
>
> * __Positive, negative, or no relationship?__ If $\hat{\Sigma}_{ij} > 0$, then features $i$ and $j$ are positively correlated, meaning that when one feature increases above its mean, the other feature also tends to increase above its mean. If $\hat{\Sigma}_{ij} < 0$, then features $i$ and $j$ are negatively correlated, meaning that when one feature increases above its mean, the other feature tends to decrease below its mean. If $\hat{\Sigma}_{ij} = 0$, then features $i$ and $j$ are uncorrelated, meaning that there is no linear relationship between the two features.
>
> * __Symmetry__: The covariance matrix $\hat{\mathbf{\Sigma}}$ is symmetric, meaning that $\hat{\Sigma}_{ij} = \hat{\Sigma}_{ji}$ for all $i$ and $j$. This follows directly from the definition of covariance.
>
> * __Positive Semi-Definite__: The covariance matrix $\hat{\mathbf{\Sigma}}$ is positive semi-definite, meaning that for any vector $\mathbf{v} \in \mathbb{R}^m$, we have $\mathbf{v}^{\top}\hat{\mathbf{\Sigma}}\mathbf{v} \geq 0$. This holds because $\mathbf{v}^{\top}\hat{\mathbf{\Sigma}}\mathbf{v}=\frac{1}{n-1}\|\tilde{\mathbf{X}}\mathbf{v}\|_{2}^{2}$, the variance of the centered data projected onto $\mathbf{v}$, which can never be negative.

### Decomposing the Covariance Matrix
Now to the magical part! The (empirical) covariance matrix $\hat{\mathbf{\Sigma}}$ can be decomposed into its eigenvalues and eigenvectors. But why is this so important? Many reasons, but let's look at a few of the most important ones. Because $\hat{\mathbf{\Sigma}}$ is real, symmetric, and positive semi-definite, its eigenpairs $(\mathbf{v}_i,\lambda_i)$ have several special and useful properties:

1. **Orthonormal eigenvectors**: Because $\hat{\mathbf{\Sigma}}$ is symmetric, its eigenvectors $\mathbf{v}_i$ can be chosen to be orthonormal, meaning that they are mutually orthogonal and have unit length. Collecting them in the matrix $\mathbf{V}=[\,\mathbf{v}_1,\dots,\mathbf{v}_m]\in\mathbb{R}^{m\times m}$ gives $\mathbf{V}^\top \mathbf{V} = \mathbf{I}$, so we can write the covariance matrix in terms of its eigenvectors and eigenvalues as:

   $$
   \hat{\mathbf{\Sigma}} \;=\; \mathbf{V}\,\mathbf{\Lambda}\,\mathbf{V}^\top,
   $$

   where $\mathbf{\Lambda}$ is the diagonal matrix of eigenvalues.

2. **Non-negative eigenvalues**: All eigenvalues $\lambda_i$ of the covariance matrix $\hat{\mathbf{\Sigma}}$ are non-negative, i.e., $\lambda_i \ge 0$. This is because the covariance matrix is positive semi-definite. The eigenvalues can be arranged in descending order:

   $$
   \mathbf{\Lambda}=\operatorname{diag}(\lambda_1,\dots,\lambda_m),
   \quad
   \lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_m \geq 0.
   $$

   Each $\lambda_i$ equals the _variance of the data_ along the direction $\mathbf{v}_i$. That seems to be a very important property, so let's unpack it a bit more.

3. **Principal components**: Subtracting the mean from a data vector $\mathbf{x}$ and projecting the result onto $\mathbf{v}_i$ gives the $i$-th principal component:

   $$
   y_i \;=\; \mathbf{v}_i^\top(\mathbf{x}-\bar{\mathbf{x}}),
   $$

   whose variance across the dataset is exactly $\lambda_i$. Thus, each eigenvalue $\lambda_i$ is the amount of variance captured by its principal component. The first principal direction $\mathbf{v}_1$ captures the most variance, the second direction $\mathbf{v}_2$ captures the second most, and so on.

4. **Optimal low-rank approximation**: Keeping only the top $k$ eigenvectors (the columns of $\mathbf{V}_k=[\mathbf{v}_1,\dots,\mathbf{v}_k]$) yields the best $k$-dimensional linear reconstruction of the data in the least-squares sense.  In other words, the subspace spanned by $\{\mathbf{v}_1,\dots,\mathbf{v}_k\}$ maximizes retained variance and minimizes reconstruction error. We'll see the matrix version of this result, the Eckart–Young theorem, in the SVD section.

These properties make the eigenpairs $(\mathbf{v}_i,\lambda_i)$ of the covariance matrix the foundation of principal component analysis (PCA).

### From Eigendecomposition to SVD
The covariance matrix approach is elegant, but it has a limitation: it only works for square, symmetric matrices like $\hat{\mathbf{\Sigma}}$. What if we want to directly factor our data matrix $\tilde{\mathbf{X}}\in\mathbb{R}^{n\times m}$, which might be rectangular (more samples than features, or vice versa)?

> **Enter SVD!** The Singular Value Decomposition provides a factorization that works for *any* rectangular matrix, not just symmetric ones. Even better, SVD applied to the centered data matrix $\tilde{\mathbf{X}}$ gives us the same principal directions as the eigendecomposition of $\hat{\mathbf{\Sigma}}$. The two approaches are mathematically equivalent, as we'll show below!

This is why SVD has become the workhorse of dimensionality reduction: it's more general, numerically stable, and directly decomposes your data. Let's explore this alternative approach.

___

## Singular value decomposition (SVD)
The singular value decomposition _factors_ a matrix into three components. 
Given a matrix $\mathbf{A}\in\mathbb{R}^{m\times n}$ of rank $r(\mathbf{A})$, the **full** SVD is given by the factorization:
$$
\mathbf{A} = \mathbf{U}\,\mathbf{S}\,\mathbf{V}^\top,
$$

where:
* $\mathbf{U}\in\mathbb{R}^{m\times m}$ is orthogonal ($\mathbf{U}^\top \mathbf{U} = \mathbf{I}_m$), and its columns $\mathbf{u}_i$ are the left singular vectors,
* $\mathbf{S}\in\mathbb{R}^{m\times n}$ is rectangular diagonal, with the singular values $\sigma_1\ge\cdots\ge\sigma_r>0$ on the first $r(\mathbf{A})$ diagonal entries and zeros elsewhere,
* $\mathbf{V}\in\mathbb{R}^{n\times n}$ is orthogonal ($\mathbf{V}^\top \mathbf{V} = \mathbf{I}_n$), and its columns $\mathbf{v}_i$ are the right singular vectors.

From here on, $\sigma_i$ denotes a singular value. The singular values are the square roots of the eigenvalues of the matrix $\mathbf{A}^{\top}\mathbf{A}$, i.e., $\sigma_i = \sqrt{\lambda_i(\mathbf{A}^\top\mathbf{A})}$, where $\lambda_i(\mathbf{A}^\top\mathbf{A})$ denote the eigenvalues of the matrix $\mathbf{A}^\top\mathbf{A}$. The number of non-zero singular values is [the rank](https://en.wikipedia.org/wiki/Rank_(linear_algebra)) of the matrix $\mathbf{A}$, where $r(\mathbf{A}) \leq\min\left(n,m\right)$.

__The full SVD__ stores **all** $m$ left‐singular vectors and all $n$ right‐singular vectors, even though only the first $r(\mathbf{A})$ correspond to nonzero singular values.

* **Pros:** You have complete orthonormal bases for $\mathbb{R}^{m}$ and $\mathbb{R}^{n}$; the first $r(\mathbf{A})$ columns of $\mathbf{U}$ and $\mathbf{V}$ span the column and row spaces of $\mathbf{A}$.
* **Cons:** Memory requirement is $\mathcal{O}(m^2 + n^2)$, which can be wasteful if $r(\mathbf{A})\ll\min(m,n)$. When you have a low rank matrix, you can save memory by storing only the first $r(\mathbf{A})$ left‐ and right‐singular vectors, which is the idea underlying the **compact SVD**.

__Compact SVD__: The compact factorization is given by $\mathbf{A}=\mathbf{U}_r\mathbf{S}_r\mathbf{V}_r^{\top}$ with $\mathbf{U}_r\in\mathbb{R}^{m\times r},\mathbf{V}_r\in\mathbb{R}^{n\times r},\mathbf{S}_r\in\mathbb{R}^{r\times r}$, where $r = r(\mathbf{A})$ is the rank of the matrix $\mathbf{A}$. Use the compact SVD whenever you only care about the nonzero singular values/directions (which is most of the time!).

__Thin SVD__: The thin SVD sits in between. It keeps $\min(m,n)$ singular vectors on each side, so it can include zero singular values when $r(\mathbf{A})<\min(m,n)$. [The `svd(...)` function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.svd) in Julia returns the thin SVD by default.

### PCA from the SVD
We can now show why PCA and the SVD of the centered data give the same principal directions.

> __Covariance eigendecomposition from the SVD__
>
> Let the centered data matrix $\tilde{\mathbf{X}}\in\mathbb{R}^{n\times m}$ have the full SVD $\tilde{\mathbf{X}}=\mathbf{U}\mathbf{S}\mathbf{V}^{\top}$, with $\mathbf{U}\in\mathbb{R}^{n\times n}$, $\mathbf{S}\in\mathbb{R}^{n\times m}$, and $\mathbf{V}\in\mathbb{R}^{m\times m}$. Substituting into the empirical covariance matrix and using $\mathbf{U}^{\top}\mathbf{U}=\mathbf{I}_n$ gives:
>
> $$
> \hat{\mathbf{\Sigma}}=\frac{1}{n-1}\tilde{\mathbf{X}}^{\top}\tilde{\mathbf{X}}=\frac{1}{n-1}\mathbf{V}\mathbf{S}^{\top}\underbrace{\mathbf{U}^{\top}\mathbf{U}}_{\mathbf{I}_n}\mathbf{S}\mathbf{V}^{\top}=\mathbf{V}\left(\frac{\mathbf{S}^{\top}\mathbf{S}}{n-1}\right)\mathbf{V}^{\top}
> $$
> where $\sigma_{i}$ are the singular values of $\tilde{\mathbf{X}}$, and $\mathbf{S}^{\top}\mathbf{S}\in\mathbb{R}^{m\times m}$ is diagonal, with $\sigma_{i}^{2}$ in its first $r(\tilde{\mathbf{X}})$ diagonal entries and zeros elsewhere. This is the eigendecomposition $\hat{\mathbf{\Sigma}}=\mathbf{V}\mathbf{\Lambda}\mathbf{V}^{\top}$: the right singular vectors of $\tilde{\mathbf{X}}$ are the eigenvectors of $\hat{\mathbf{\Sigma}}$, and the eigenvalues are $\lambda_{i}=\sigma_{i}^{2}/(n-1)$, taking $\sigma_{i}=0$ for $i>r(\tilde{\mathbf{X}})$.

Thus, we can compute the principal directions and their variances from the SVD of $\tilde{\mathbf{X}}$ without forming $\hat{\mathbf{\Sigma}}$.

### Dimensionality Reduction?
The SVD decomposes any rectangular matrix into a weighted sum of rank-one matrices, making it ideal for dimensionality reduction and data compression. Let $\mathbf{A}\in\mathbb{R}^{m\times{n}}$ have the singular value decomposition $\mathbf{A} = \mathbf{U}\mathbf{S}\mathbf{V}^{\top}$. Then, the matrix $\mathbf{A}$ can be written as:
$$
\mathbf{A} = \sum_{i=1}^{r(\mathbf{A})}\sigma_{i}\,\underbrace{\left(\mathbf{u}_{i}\otimes\mathbf{v}_{i}\right)}_{\mathbf{u}_{i}\mathbf{v}_{i}^{\top}}
$$
where $r(\mathbf{A})$ is the rank of matrix $\mathbf{A}$, the vectors $\mathbf{u}_{i}$ and $\mathbf{v}_{i}$ are the $i$-th left and right singular vectors, and $\sigma_{i}$ are the (ordered) singular values. The [outer product](https://en.wikipedia.org/wiki/Outer_product) $\mathbf{u}_{i}\mathbf{v}_{i}^{\top}$ is an $m\times n$ matrix of rank one, so each term in the sum is a separable rank-one component of $\mathbf{A}$ weighted by $\sigma_{i}$.

> __Eckart–Young theorem__
>
> Let $\mathbf{A}\in\mathbb{R}^{m\times n}$ have rank $r=r(\mathbf{A})$, singular values $\sigma_1\ge\sigma_2\ge\cdots\ge\sigma_r>0$, and left and right singular vectors $\mathbf{u}_i$ and $\mathbf{v}_i$. Truncating the sum after $k<r(\mathbf{A})$ terms gives the rank-$k$ matrix:
>
> $$
> \mathbf{A}_k = \sum_{i=1}^k \sigma_i\,\mathbf{u}_i\,\mathbf{v}_i^\top.
> $$
>
> For any other matrix $\mathbf{B}\in\mathbb{R}^{m\times n}$ of rank at most $k$,
>
> $$
> \|\mathbf{A} - \mathbf{A}_k\|_F \;\le\;\|\mathbf{A} - \mathbf{B}\|_F,\quad\|\mathbf{A} - \mathbf{A}_k\|_2 \;\le\;\|\mathbf{A} - \mathbf{B}\|_2,
> $$
>
> where $\|\cdot\|_F$ is the Frobenius norm and $\|\cdot\|_2$ the operator (spectral) norm. In other words, $\mathbf{A}_k$ is the best rank-$k$ approximation to $\mathbf{A}$ under both measures of error.

Thus, truncating the SVD isn't just a heuristic, it's *provably* the closest rank-$k$ matrix to $\mathbf{A}$, whether you measure closeness by the square root of the total squared error (Frobenius) or by maximum stretch (spectral).

Let's look at an example of the truncated SVD in action.

> __Example__
>
> [▶ Fun with Singular Value Decomposition](CHEME-5800-L7a-Example-FunWithSVD-Fall-2026.ipynb). What does the SVD reveal about the structure of an image? We factor a grayscale photograph into rank-one frames, each weighted by a singular value, and rebuild the image from the first few frames to see how quickly the relative error falls. We also check that the sum of the first few frames has rank equal to the number of frames added.

___

## Lab
In [L7b](../L7b/CHEME-5800-L7b-Lab-SVD-StoichiometricMatrix-Fall-2026.ipynb), we will take a deeper dive into singular value decomposition (SVD). We will construct a stoichiometric matrix from a human platelet metabolic model and use its SVD to examine numerical rank, matrix reconstruction, conservation relations, and balanced-flux directions.

___

## Summary
We developed the eigendecomposition of the empirical covariance matrix and the singular value decomposition, and showed that both give the same principal directions for centered data.

> __Key Takeaways:__
>
> * __Covariance eigendecomposition:__ We built the empirical covariance matrix from the centered data matrix and decomposed it into orthonormal eigenvectors and non-negative eigenvalues. Each eigenvalue is the variance of the data along its eigenvector, which ranks the directions for dimensionality reduction.
>
> * __Truncated SVD:__ We factored a general matrix into two orthogonal matrices of singular vectors and a rectangular diagonal matrix of singular values, and compared its full, compact, and thin forms. Keeping only the largest singular values gives the closest matrix of that rank in the Frobenius and spectral norms, as the Eckart–Young theorem states.
>
> * __PCA from the SVD:__ We substituted the SVD of the centered data matrix into the covariance matrix and found that its right singular vectors are the covariance eigenvectors, with eigenvalues proportional to the squared singular values. This lets us compute the principal directions and their variances without forming the covariance matrix.

The example applies the truncated SVD to a photograph, and the L7b lab applies the SVD to a metabolic stoichiometric matrix.
___